# 模块化迁移系统整合开发记录

日期：2026-11-16。范围：工程整合与本地校准，不是新一批模型实验或独立测试成绩。

依据开题报告研究内容 1.1—1.4，将现有实验能力接入 `migration_system`：真实 LangGraph 主图、SSM、few-shot 生成、静态／AST TED／DOM／Puppeteer／SSIM／像素／样式／代表性交互、规则与模型修复、候选回退、API／数据库／工作台及人工台账。LangSmith SDK 摘要追踪、LoRA 数据准备／训练／adapter 推理及本地视觉嵌入有实际接入代码；外部资源缺失时明确不执行。

实现路径、命令、边界与最终内测结果以仓库根目录 `migration_system/SYSTEM_GUIDE.md`、`SETUP_PENDING.md`、`DEVELOPMENT_20261116.md` 为准。保留旧 Notebook、快照、Vue 源码及旧夹具，不覆盖历史结论。

## 新协议与不能混淆的结果

新协议 `modular-migration-v1` 最多一次生成、三轮修复。次数、每次输出上限和已知 usage 停止线不是硬计费上限。旧 G3 上界批准与旧请求保持原状，网络默认关闭；启用后创建任务仍需明确确认调用政策。

源基准警告不抑制；目标异常与设施故障分流；失败候选不可覆盖接受版本；未知模型调用不重发。人工结果绑定产物哈希、另行追加，不覆盖自动分数。

结构、视觉、交互与编译运行分别报告。UI = 100 × (0.5×SSIM + 0.3×(1−不同像素率) + 0.2×样式一致率)。结构与 UI 阈值 >95%，不同像素率 <5%。未配置交互标为未测，不代表所有行为等价；新口径不能与旧实验直接合并。

本轮没有云模型调用、LangSmith 上传、模型下载、LoRA 训练或历史 30 项复跑。默认 few-shot 是两个自建开发演示，不是测试集答案，不据此宣称准确率提升。

## 本地真实证据与部署状态

真实 Puppeteer 校准包含：同图、已知颜色差异、目标运行异常、SSM＋旧静态校验＋按钮交互，以及实际 Vue 工作台→Flask→临时 SQLite 提交。图控制／恢复／费用未知／回退等用明确标记的测试客户端，不将其结果冒称模型产物。

用户实际业务库只读核对为 `0006_generation_approvals`，三条源输入；原 SSM 的 request／receipt／ssm 三份 SHA-256 未变。新代码新增 0007 四表，未自动迁移实际库。统一部署需显式 init-db 后启动 API 与 migration-worker。

后续真实模型、LangSmith 与 LoRA 的外部验收应另追加记录，不回填为本轮已完成。

## 内部验证结果

全套当次 330 项通过（595.297 秒）；当前模块收口专项 32 项通过（59.724 秒），训练数据专项 3 项通过。当前收集为 333 项，新增用例包含在专项中，没有声称用同一命令一次跑过全部 333 项。专项与全套重叠，不能相加当样本数。

原 P1 40 项、前端 Node 22 项、像素 Node 1 项通过；TypeScript 与 Vite 构建通过。真实浏览器校准与真实前端临时库提交通过。源码／锁文件／日志哈希摘要见 `migration_system/verification_20261116.json`。

收口边界包括：被拒绝规则转模型修复、规则轮次不占模型调用名额、超限响应保留用量且不能借收据重放放行、嵌套 template 不截断 SFC。上述是工程控制与固定样例证据，不是模型迁移成功率。

## 首次真实提交阻断与 CommonJS 源加载修复

用户启用模型网络后提交 DeliveryPhaseCard.vue。任务 `3a5205caedf0413493256822a1928be9` 在 preflight 阻断：`unsupported_source_sfc`，模型开始 0 次、未知调用 0 次、已知 Tokens 0。上传源码与仓库原文件 SHA-256 均为 `2a057ec6591b35d3044b9b77106e849ccea48e0ac407ab39d34d7e37fceda16b`，不是源码损坏。

根因是新浏览器适配器仅支持 export default，未兼容原样本的 module.exports Options 对象。修复位于 browser/evaluate.mjs 的源加载适配层，不涉及生成／修复 Prompt。仅支持单一顶层 module.exports = { ... }，浏览器内提供模块作用域；混合／重复导出、工厂调用和外部 import 仍阻断。未修改原 Vue、上传文件或旧任务证据，也未降低阈值。

新增两种导出预检正例、歧义／依赖阻断反例、真实浏览器 CommonJS 点击更新测试。测试夹具强制关闭模型网络，不继承用户启用状态。重启 API／migration-worker 后需新建真实任务；旧阻断任务不能改绑成修复后通过。验证结果另见 migration_system/DEVELOPMENT_20261116.md。

## FocusCard 评估误判修正与新旧离线对照

原真实任务 f3cf07b0c7e5462aaf1b1cd5a781b37a 为 failed / repair_budget_exhausted，模型开始 3 次、已知 Tokens 41391。最终接受的是第 1 轮产物，SHA-256 为 0ced63ec865d38bedf3cdc4d5a102780d35d48684639af75900675f69decaced。原报告静态 class fail、SSM/San 结构 57.14%，实际 DOM／视觉均 100%。

修正共享旧 class 检查器对条件字符串空白的误判，保持 token 边界和完整条件验证；修正新 AST 投影对 San 三元插值、固定 class、相邻文本分段的处理，投影版本为 v2；样式路径按元素兄弟编号，避免缩进空白导致错位。阈值、TED 代价、像素与 UI 公式不变，未修改生成／修复 Prompt，也未手改生成代码。结构差异附具体路径／标签；依赖指纹覆盖实际 Node 检查器。

用原字节产物离线重评：旧验证器直接调用与新适配层均 pass；SSM/San 与源/San AST TED 均为 0，DOM 100%，UI 100%；旧 VisualEvalStage 的 DOM／文本相似度也均为 100%（不冒充像素评估）。另行补充开始／停止两次点击，代表交互 pass。历史空场景仍 not_run，不回填。

本轮新增模型调用 0。原失败任务及其用量不变，离线重评另存为 data/experiments/evaluator_calibration/2026-11-16_modular_focus_card_evaluator_replay_v1；补齐多 token 源 class 键的空白边界后，最终实现另存为同级的 2026-11-16_modular_focus_card_evaluator_replay_v2，不覆盖 v1。包含历史索引核验、原报告／源码／产物副本、新截图、comparison.json 和 SHA256SUMS.json。它是开发校准，不是独立测试或新模型成功样本；测试执行结果见 migration_system/DEVELOPMENT_20261116.md。

## ChoiceChips：实际 DOM 诊断与空 class 受限比较

原任务 6d2eb1d3cc034c88b1b4167728334c8c 为 failed / repair_budget_exhausted，模型开始 3 次、已知 Tokens 36725。接受产物是第 2 轮，SHA-256 为 6b4bd4b84d866d93e37a0fdbb742ea3d6ea2ce52d39a22088a1c4a43121bce29；原报告 SHA-256 bb45e4d3745e03fdfaf8acc5d4e31a7b79c53fc6bcc07bc6055abe717322994a。SSM/San AST 已为 100%，原始 DOM 83.33%，差异为两个未选中按钮的空 class／缺失 class。原反馈未给实际 DOM 差异，导致无明确定位的修复消耗。

补齐 DOM／动作失败的路径和属性诊断，并在候选保护不退化的前提下识别 DOM 严格改善。新增 empty_class_presence_v1 受限比较策略：双端保守 AST／模板／CSS 检查获准后，仅在比较副本里归一化精确空 class；存在属性选择器、DOM 观察、反射、未知依赖或分析失败则保留严格比较。原始树、原始 TED、策略原因和改变路径全部保留，不证明任意外部 DOM 观察者等价。

原产物离线重评：静态、编译运行、源基准、SSM/San 结构、视觉通过；原始 DOM 83.33% 不变，有效比较 100%，UI 100%。另补充即时、每日、关闭、恢复每日四次选择，均通过。辅助源模板 AST 87.5% 保留，不回填历史空场景 not_run。没有修改生成代码、Prompt 模板、阈值或预算，实际修复消息会包含新增诊断。

本轮新增模型调用 0。原失败任务和 36725 Tokens 账本不变。46 条历史哈希引用（含重复引用）核验后，独立证据存于 data/experiments/evaluator_calibration/2026-11-16_modular_choice_chips_dom_replay_v1，包含原报告／源码／接受产物副本、截图、comparison.json、SHA256SUMS.json 和 REPORT.md。属于开发校准，不是新模型成功样本或独立测试；测试结果见 migration_system/DEVELOPMENT_20261116.md。

## ExpenseTracker：语法优先闭环与有边界的通用修复

原任务 5c2fc8916fcd4b8bb65f32ea28e20488：blocked / model_usage_stop_reached，模型开始 3 次，已知 Tokens 175512。Vue 源哈希 47995c773b5698e5f4fda5b5e854b27648c3b1e030a845ab19790930ad564736。不是已证明 Vue 源码错误：生成的 San 使用无效 <s-for=…> 包装，前两轮脚本方法之间缺逗号。适配层只返回 invalid_target_sfc，语义分析级联，未优先修好语法。

新增 target_syntax_first_v1：SFC／Babel／San 详细诊断，行列／片段及位置精度；语法失败时推迟语义、双端浏览器、结构与视觉检查。新增两条无组件名白名单的局部规则：源码唯一匹配循环／key／元素后把伪包装改为单子元素的 s-for 属性；仅在 Babel 指向缺逗号且完整新 AST 确认相邻对象方法边界时补一个逗号。字符串／注释／插值、未知来源、多子元素、嵌套包装及丢失元素不猜测修复。

旧生成 Prompt、旧语义修复 Prompt 入口保留；新增聚焦语法消息分支，只传当前目标、原 Vue 与具体解析错误，不重复完整 SSM。规则候选必须复测；语法首次通过后新增的语义观测不会误当成回退，仍须所有必需维度通过才能判成功。已通过维度／源基准保护不取消，预算和质量阈值未放宽。

离线重放先复制并核对 24 个历史任务文件及源输入（共 25 文件），无模型调用、不更新任务。最终证据 data/experiments/evaluator_calibration/2026-11-16_modular_expense_tracker_syntax_replay_v4。初始版诊断 10→3，因缺 option 子元素继续阻断；第 1／2 轮产物诊断分别 10→0、9→0，真实编译运行通过，但静态语义／DOM 仍失败；代表交互仍 not_run。不得把语法改善写成整个样本迁移通过。

v1 保留沙箱内浏览器超时的未完成现场，v2／v3 为中间完整证据，v4 对应最终位置精度标注。原失败和 175512 Tokens 不抹除，未改原 San、源 Vue、用户配置或手工测试页。最终模块回归 70 项通过、166.829 秒，含新增语法专项 16 项；工程用例不当作迁移成功率分母。30 个证据 payload 与实现指纹核验通过。聚焦语法消息尚未发送真实模型，不宣称模型修复成功率已验证。规则、测试及部署细节见 migration_system/SYNTAX_REPAIR.md、DEVELOPMENT_20261116.md。这是开发回归，不是独立测试或新模型成功样本。

## ExpenseTracker：语法之后的共享规则／验证闭环

继续使用原失败任务 5c2fc8916fcd4b8bb65f32ea28e20488 的接受 revision 1，经上一轮语法规则得到的固定输入 6b513daf6bb3e8a12738516f01d2c8b34bd0be6f675171dee84f1fea2065438a，不重新生成或手改原产物。Enter 事件复用旧规则；补齐类名字符串分隔、数值样式对象／CSS 插值的来源证明及公开签名恢复、已有 watcher 时 mounted 调用移动到其后的阶段规则。修改位于 migration_pipeline/utils 的共享实现，新系统通过 residual_v2 适配入口复用，而非组件名特判。

规则有明确边界：只证明匹配引用的类名分隔符，不证明任意动态类型转换；合法字符串拼接不误报。样式只支持单尺寸属性、显式匹配单位、可证明数值只读叶方法，保留源公开调用方式并隔离模板依赖；未知调用、脚本／多模板引用、单位或方法体变化不自动修改。watcher 字段、选项和无参回调必须匹配源，无其他早期／未知启动行为，不复制加载调用。

离线候选 SHA-256 9b8be8e3a0adcd7bd1f0b03246601b21441954a281b281ad31acea542fd1e34f：当前静态错误 7→0，编译运行／源基准／静态／结构／视觉通过，原始及有效 DOM、SSM/San 结构均 100%，UI 100%、SSIM 1、不同像素 0。辅助源 Vue／目标模板 AST 保留 96.10%（三个 binding:key／trackby 表示位置差异），未改为 100%。门控 strict_improvement 接受，规则重复运行不再改动字节。另加非 Enter 不添加、Enter 添加、收支切换、过滤／清除、主题及删除共 15 步诊断交互全部通过；旧任务的交互 not_run 不回填。

证据 data/experiments/evaluator_calibration/2026-11-16_modular_expense_tracker_semantic_replay_v3：先核对并冻结上轮 30 个 payload，保留实现、候选、双端截图、逐步原始快照、差异与门控。v1／v2 为中间证据，不覆盖。原失败状态和 175512 Tokens 账本保留，本轮模型调用 0、无 LangSmith 上传、未改用户配置／原 Vue／原 San／手工测试页，未修改 Prompt 或增加预算。

最终分别验证新增规则 16 项通过（41.386 秒）、模块 70 项通过（320.199 秒）、旧 TaskManager 冻结闭环独立专项 1 项通过（186.200 秒）。开发中未全通过的中间合并运行不算通过证据，夹具修正和复测说明见 DEVELOPMENT_20261116.md。不把工程测试数量当迁移样本数，也不把冻结开发产物的离线候选通过计为新模型或独立测试成功。真实模型端到端、任意输入／全部公开调用和跨刷新持久化不在本次证明范围；部署需重启 API／Worker 后新建任务。

In [ ]:
# 只读检查；不初始化数据库、不启动 Worker、不调用模型、不打印密钥。
from pathlib import Path
import subprocess
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'migration_system').is_dir())
python_exe = root / 'migration_system/.venv/Scripts/python.exe'
result = subprocess.run([str(python_exe), 'migration_system/scripts/run_system.py', 'system-status'], cwd=root, capture_output=True, text=True, timeout=30)
print(result.stdout or result.stderr)

## 2026-11-17：从本次初始生成出发的完整规则闭环

原任务 b2a59e959191487fa4c3acbe945f04cb（ExpenseTracker）仍为 failed / repair_budget_exhausted，真实模型调用 3 次、已知 Tokens 80802，不改历史结果。初始语法诊断 10 条，第 1 轮旧规则降为 4 条；后两次模型请求 messages 相同，候选均无进展而被拒绝。这不是视觉分数或源 Vue 错误。先冻结 29 个任务文件与源输入，共 30 文件；源码 SHA-256 47995c773b5698e5f4fda5b5e854b27648c3b1e030a845ab19790930ad564736。

本轮扩展来源可证明的缺失 option、多处方法逗号／闭合标点恢复；保留已接受父版本，附带同父版本拒绝候选的哈希、原因和实际诊断；重复完整修复 messages 在模型调用前阻断。生成 Prompt 和旧语义 Prompt 模板未改，语法专用消息新增拒绝反馈，预算／Token 停止线／阈值不变。规则不看组件名，不手改生成文件。

完整重放进一步暴露字符串 CSS 包络、非 San watcher 配置和 missing_watch 诊断未调度对应规则的遗漏；已同时补共享 CJS、RepairStage._deterministic_repair_plan 的触发映射及正反例。watcher 转换只支持源明确 deep:true 的精确零参回调和来源匹配启动序列，浅 watcher、immediate、未知副作用继续阻断。

语法最终证据 data/experiments/evaluator_calibration/2026-11-17_modular_expense_tracker_syntax_closure_v3：4 个冻结 revision 均通过语法，36 payload。使用其初始 revision 0 的规则输出继续语义重放，最终证据 data/experiments/evaluator_calibration/2026-11-17_modular_expense_tracker_semantic_closure_v3：诊断总数 9→0（包含结构诊断，不全是静态错误），静态／编译运行／源基准／结构／视觉通过，SSM／San、原始与有效 DOM、UI 均 100%，SSIM 1、不同像素 0；另加固定 15 步动作全部通过，不回填原空动作。辅助源／目标 AST 仍 96.10%，未修改指标。候选门控接受、规则幂等，最终候选 SHA-256 0c0d2cde569db4f1fce50b680da732a2b7888e06ce90b19109bfa660e9e00ff3；新证据封存 62 payload。

最终完整模块回归 79 项通过（490.876 秒），含初始历史响应／SSM注入临时数据库后，真实 LangGraph→规则→候选门控→浏览器 15 动作的路径；额度仍为 3 轮，修复 revision 全来自规则，没有额外模型修复。生成调用使用明确测试响应，不用测试替代的评估分数。共享语义／实际调度专项 20 项通过（75.513 秒）；原流水线回归 78 项通过（383.318 秒）；语法／反馈专项 24 项通过（53.290 秒，已包含在模块回归）。这些是有重叠工程测试，不相加为迁移样本分母。

保留 syntax v1／v2 和 semantic v1／v2 中间结果，后两者静态仍剩 3／2 条，不计成功；受限环境浏览器启动失败也不计质量通过。原任务与源文件哈希未变，本轮新增真实模型调用 0、LangSmith 上传 0，不修改真实数据库、配置或手工页。这是开发失败的离线闭环，不是新模型迁移成功或独立测试证据；未证明跨刷新持久化、任意 props／全部公开调用等价。部署需重启 API 与 migration-worker，然后新建任务；不可恢复旧任务改写历史。细节见 migration_system/SYNTAX_CLOSURE_20261117.md 和 DEVELOPMENT_20261116.md。

## 2026-11-17：源模板 HTML 空元素解析入口修复

原任务 3d3d69c7760045e1bd0c741f7cf3638b（InvoiceLineEditor.vue）为 blocked / browser_evaluation_blocked，模型调用 0、Tokens 0。源源码 SHA-256 9b50005a20314dd9c1f52db9496c6092d8f2838958955be0fa0350db7c77a8df。新评估器的 Vue 模板 AST 入口使用 compiler-dom.baseParse，缺少 HTML 平台空元素规则，误将合法 input 识别为缺少结束标签。SFC 解析、HTML 模板解析及编译的独立对照均通过，不将其归因为源代码或模型问题。

源投影改为 compiler-dom.parse；San 目标解析仍保持原入口，不加忽略错误的回调或正则补标签。input／img／br／hr／wbr 的普通和自闭合写法均覆盖；div／span／textarea／strong 等真实缺失或错配结束标签继续生成前阻断。新增源模板／SFC／脚本错误代码，报告携带绝对 UTF-16 位置、1-based 行列、定位精度和有限源码片段；Python 按已知源码重算，过滤外来字段。工作台转义显示片段，模型未调用时明确提示，LangSmith 不上传源码诊断。

先冻结原 request、failure 与源输入，共三个文件；修复前证据 data/experiments/evaluator_calibration/2026-11-17_modular_source_void_elements_preflight_v1，修复后为同级 v2。源码字节不变，源 preflight 从 blocked 变为 completed；v2 10 个 payload、当前实现指纹及三个原件哈希均核验。重放不启动渲染浏览器、不调用模型、不修改真实数据库或历史任务；没有生成该组件的 San，源预检查通过不能记为迁移质量通过。

专项 12 项通过（33.895 秒），含真实解析器、临时数据库／LangGraph 零调用阻断、实际工作台诊断显示及双端浏览器；完整模块回归 91 项通过（241.352 秒），包含专项，不相加为迁移样本数。前端 TypeScript／Vite 构建和 Node 22 项测试通过。本轮未重跑根目录旧流水线全套。开发中静态 input value 的框架属性表示差异曾使夹具失败，改用 placeholder 隔离空元素解析后重新验证，未放宽 DOM 比较规则；该独立属性差异不宣称已修复。

未修改 SSM、生成／修复 Prompt、预算、阈值、用户配置或手工页，未删除适配器。原任务和费用原样保留；新增真实模型调用 0、LangSmith 上传 0。这是工程回归与开发校准，不是新的模型迁移成功样本或独立测试。记录见 migration_system/SOURCE_TEMPLATE_PARSING.md 和 DEVELOPMENT_20261116.md；部署重启 API 与 migration-worker、刷新页面后新建任务，无需 init-db，后续真实迁移仍需单独验证。

## 2026-11-17：ProductFilterPanel 任意公开键能力边界与控件实际值诊断

原任务 ea3808cfe9ff4ae7b301e5c8d593e3aa：failed / repair_budget_exhausted，模型 3 次（生成一次、模型修复两次；另有一轮规则修复），Tokens 87999。共享规则补齐 props 默认值及循环，模型恢复混合节点顺序，最后一次响应与接受 revision 2 完全相同而被拒绝。源语法、编译运行和最终视觉通过；静态剩 source_dynamic_assignment_unproven，原始 DOM 8 处 value／空 class 属性差异，交互原配置为空，not_run 不改写。源 SHA-256 aed0d49a2bab60b3103e50beb0236320d35f7fb9d8bd2341db36a65be65e817a。

保留用户既定的任意公开键契约。this[key]=value 与 this.data.set(key,value) 对字面点号键、Symbol、实例成员命名空间不能直接证明等价；原生 San 反例确认点号键为路径而非字面实例属性。没有把 setter 降级为六个固定键，也没有开发完整动态实例兼容层，故本轮不宣称语义能力已修好。共享 method_interface_semantics 导出原识别器，新源能力检查复用它，在语法检查后、SSM／模型调用前以 source_dynamic_key_contract_unsupported 明确阻断，源代码合法与解析错误分开提示。未知方法形状仍交由原验证器，不把未匹配视为完整支持证明。

初始态新增 live value／checked 检查，多选 select 采集全部选中值；状态不同即产生 rendered_control_state_mismatch 并使结构失败。初始与动作后的实际值差异写入报告，原 DOM 不改写，值相同不能豁免 value 属性／class 存在性。空 class 分析中的另一处 baseParse 改为 HTML parse，但动态成员／DOM 反射／未知调用继续拒绝归一化，ProductFilterPanel 未获批准。工作台新增原始诊断和初始控件状态显示，不改阈值或 Prompt 模板。

先冻结 45 个原任务文件、源输入和 freeze.json，47 payload，目录 data/experiments/evaluator_calibration/2026-11-17_modular_product_filter_failure_frozen_v1。最终独立证据为同级 2026-11-17_modular_product_filter_contract_replay_v4，21 payload，使用原接受 revision 2、未手改生成文件。11 步限定动作的控件实际值全部一致，双端含初始态 24 条明确商品列表预期全部符合；但严格 DOM 继续失败，新增场景的整体 interaction 为 fail，static 仍 fail，passed=false。不能把限定 UI 观察等同于任意公开方法或独立迁移成功。

v1 为受限环境浏览器失败的未完成现场，v2 因场景预期断言失败退出；v3 保存错误预期下的完整观察。初始预期误用输入数组顺序，并假设 clear 重置 sort；v4 按源实际名称排序及 clear 保留已选排序修正夹具后重测。生产代码／源代码没有为夹具而修改。

最终新增专项 11 项通过（33.120 秒），完整模块 102 项通过（671.729 秒，已含专项），原共享方法接口 16 项通过（463.020 秒）；前端构建和 Node 22 项通过，不相加为迁移样本数。初次专项模块路径错误、受限环境浏览器失败及其修正／最终完整复测独立记录，根目录其他测试未全量重跑。47 个冻结 payload、21 个最终重放 payload、当前实现和 46 个原件哈希核验。部署细节见 migration_system/DEVELOPMENT_20261116.md 与 PUBLIC_KEY_CONTRACT_20261117.md。原失败、账本和 not_run 保留；本轮真实模型调用 0、LangSmith 上传 0，无真实数据库写入、commit 或 push，不计论文迁移通过率。该组件仍需完整语义能力实现或另获批准的新契约，不能仅靠重启或增预算复跑放行。